# 03 — Top-Down Trend Scanning\n\nScan the KB-Trends pool for technology megatrends relevant to regulatory frequency monitoring.\nFor each trend, assess its specific impact on the domain.\nOutput: Trend-derived Technology Drivers.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR, MAX_RAG_CHUNKS
from src.llm import embed, safe_chat_json
from src.traceability import TechDriver, DriverOrigin, DriverConfidence, KBPool, stable_id
from src import prompts

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

print(f"ChromaDB: {collection.count()} documents")

ChromaDB: 2850 documents


In [2]:
def retrieve_trends(query: str, n: int = MAX_RAG_CHUNKS) -> list[dict]:
    query_emb = embed([query])[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n,
        where={"pool": "trend"},
        include=["documents", "metadatas"],
    )
    out = []
    for i in range(len(results["ids"][0])):
        out.append({
            "chunk_id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "source_title": results["metadatas"][0][i]["source_title"],
        })
    return out


def format_rag_chunks(chunks: list[dict]) -> str:
    parts = []
    for c in chunks:
        parts.append(f"[Chunk ID: {c['chunk_id']}] (Source: {c['source_title']})\n{c['content']}")
    return "\n\n---\n\n".join(parts)

## Step 1: Scan for Technology Trends

In [3]:
# scan across multiple queries to cover different trend angles
scan_queries = [
    "AI machine learning spectrum monitoring future trends",
    "quantum sensing RF detection new technology",
    "6G spectrum management cognitive radio",
    "space satellite spectrum monitoring",
    "edge computing distributed sensor networks",
]

all_trends = []
all_source_chunk_ids = []

for query in scan_queries:
    rag_chunks = retrieve_trends(query, n=4)
    rag_text = format_rag_chunks(rag_chunks)

    prompt = prompts.TREND_SCAN.format(rag_chunks=rag_text)
    result = safe_chat_json(prompt, system="You are a technology foresight analyst identifying trends relevant to regulatory spectrum monitoring.")

    chunk_ids = result.get("source_chunk_ids_used", [])
    for trend in result.get("trends", []):
        trend["source_chunk_ids"] = chunk_ids
        all_trends.append(trend)

print(f"Found {len(all_trends)} raw trends\n")
for t in all_trends:
    print(f"  [{t.get('timeframe', '?')}] {t['name']}")
    print(f"    {t['relevance'][:100]}...")
    print()

Found 29 raw trends

  [mid-term (5-10y)] AI-Driven Autonomous Spectrum Monitoring
    This trend allows regulatory bodies to continuously and proactively monitor spectrum usage, confirm ...

  [mid-term (5-10y)] Integration of Big Data and Distributed Monitoring Networks
    Enables real-time, high-quality spectrum occupancy data collection over wide areas, supporting accur...

  [mid-term (5-10y)] Radio Frequency Machine Learning (RFML) for Signal Classification and Anomaly Detection
    Enhances the capability of regulators to identify unauthorized or interfering signals, adapt to emer...

  [long-term (10-15y)] Co-location of Spectrum Sensing with Network Infrastructure
    Facilitates more granular and localized spectrum monitoring, reduces deployment costs, and allows re...

  [mid-term (5-10y)] Standardization and Interoperability of Spectrum Monitoring Data
    Critical for scaling spectrum monitoring systems across multiple sites and sensor types, ensuring co...

  [mid-term (

## Step 2: Assess Impact on Regulatory Frequency Monitoring

In [4]:
import re

def normalize_name(name):
    name = name.lower().strip()
    name = re.sub(r'\s*\([^)]*\)\s*', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()

# deduplicate trends by normalized name
seen_names = set()
unique_trends = []
for t in all_trends:
    key = normalize_name(t["name"])
    if key not in seen_names:
        seen_names.add(key)
        unique_trends.append(t)

print(f"After dedup: {len(unique_trends)} unique trends (from {len(all_trends)} raw)\n")

# assess impact for each
trend_drivers: list[TechDriver] = []

for trend in unique_trends:
    rag_chunks = retrieve_trends(f"{trend['name']} {trend['description']}", n=3)
    rag_text = format_rag_chunks(rag_chunks)

    prompt = prompts.TREND_IMPACT.format(
        trend_name=trend["name"],
        trend_description=trend["description"],
        rag_chunks=rag_text,
    )
    result = safe_chat_json(prompt, system="You are assessing how technology trends impact regulatory frequency monitoring.")

    impact = result.get("impact_level", "none")
    print(f"  [{impact.upper():6s}] {trend['name']}")
    print(f"    → {result.get('impact_description', '')[:100]}")

    if impact in ("high", "medium"):
        driver = TechDriver(
            id=stable_id(trend["name"], "trend"),
            name=trend["name"],
            description=f"{trend['description']}. Impact: {result.get('impact_description', '')}",
            origin=DriverOrigin.TREND,
            confidence=DriverConfidence.LOW,
            source_chunk_ids=trend.get("source_chunk_ids", []),
        )
        trend_drivers.append(driver)

print(f"\n=== {len(trend_drivers)} Trend Drivers (high/medium impact) ===")

After dedup: 29 unique trends (from 29 raw)



  [HIGH  ] AI-Driven Autonomous Spectrum Monitoring
    → AI-driven autonomous spectrum monitoring significantly enhances regulatory frequency monitoring by e


  [HIGH  ] Integration of Big Data and Distributed Monitoring Networks
    → The integration of big data and distributed monitoring networks significantly transforms regulatory 


  [HIGH  ] Radio Frequency Machine Learning (RFML) for Signal Classification and Anomaly Detection
    → Radio Frequency Machine Learning (RFML) significantly enhances regulatory frequency monitoring by en


  [HIGH  ] Co-location of Spectrum Sensing with Network Infrastructure
    → Co-location of spectrum sensing with network infrastructure significantly enhances regulatory freque


...

  [HIGH  ] Edge and Local Processing with RF Machine Learning (RFML)
    → Embedding RF Machine Learning (RFML) at the edge enables near real-time, autonomous spectrum sensing


  [HIGH  ] Advanced Geolocation Techniques Using Distributed Sensors
    → Advanced geolocation techniques using distributed sensors, such as TDOA, AOA, and hybrid methods, si


  [HIGH  ] Open Spectrum Data as a Service with APIs
    → The trend of Open Spectrum Data as a Service with APIs, exemplified by the Electrosense project, sig


  [HIGH  ] Enhanced Data Security and Privacy Measures Including Blockchain
    → The trend of enhanced data security and privacy measures, including blockchain, significantly improv


  [HIGH  ] Integration of Mobile and Portable Spectrum Monitoring Systems
    → The integration of mobile and portable spectrum monitoring systems significantly enhances regulatory

=== 29 Trend Drivers (high/medium impact) ===


In [5]:
# save state
trend_state = {
    "trend_drivers": [d.model_dump(mode="json") for d in trend_drivers],
    "all_trends_raw": unique_trends,
}
with open("../data/outputs/trend_state.json", "w") as f:
    json.dump(trend_state, f, indent=2)

print(f"Saved {len(trend_drivers)} trend drivers\n")
for d in trend_drivers:
    print(f"  • {d.name}")
    print(f"    {d.description[:120]}...")
    print()

Saved 29 trend drivers

  • AI-Driven Autonomous Spectrum Monitoring
    The use of artificial intelligence and machine learning to enable fully autonomous, 24/7 spectrum monitoring systems cap...

  • Integration of Big Data and Distributed Monitoring Networks
    Deployment of dense, distributed sensor networks generating large volumes of heterogeneous spectral data processed local...

  • Radio Frequency Machine Learning (RFML) for Signal Classification and Anomaly Detection
    Application of deep learning and machine learning techniques specifically tailored for radio frequency data to classify ...

  • Co-location of Spectrum Sensing with Network Infrastructure
    Embedding spectrum sensing capabilities within existing network elements such as base stations and radio units to enable...

  • Standardization and Interoperability of Spectrum Monitoring Data
    Development and adoption of common rules, protocols, and standardized data formats for RF spectrum sensing data to enabl..